In [1]:
import os
import random
import pandas as pd
import json
from tqdm import tqdm

# Change it to your google drive path where this notebook located.
drive_path = '/Users/samyiin/Projects/ZipfLawAnalysis'
os.chdir(drive_path)

This notebook is not used! We use a multiprocess runner to run the python script version of this notebook. 

In [20]:
def search_for_user_repos(user_login, github_request_machine):
    list_df_repos = []
    page_num = 0
    while True:
        df_repos = github_request_machine.user_service_endpoint(user_login=user_login, service='repos', page_num=page_num, 
                                                                retry_after_rate_limit=True)
        # it is possible that a user just have exactly 200 repos. Then the final page have 100 but there is no next page.
        if len(df_repos) == 0:
            break
        list_df_repos.append(df_repos)
        if len(df_repos) < 100:
            break
        page_num += 1
    df_repos = pd.concat(list_df_repos, ignore_index=True)
    return df_repos

def store_single_user_repo(user_login, github_request_machine):
    user_directory_path = os.path.join('Database/UserData/', user_login)
    os.makedirs(user_directory_path, exist_ok=True)
    # construct the csv file
    repo_csv_path = os.path.join(user_directory_path, 'repos.csv')
    # "Cache" the file
    if os.path.exists(repo_csv_path):
        return
    df_repos = search_for_user_repos(user_login, github_request_machine)
    df_repos['user_login'] = user_login
    # save to csv
    df_repos.to_csv(repo_csv_path, index=False)

def store_users_repos(df_users, github_request_machine):
    for i in tqdm(range(len(df_users))):
        user_login = df_users.iloc[i]['login']
        try:
            store_single_user_repo(user_login, github_request_machine)
        except:
            print(f"Trouble: {i} {user_login}")
            continue

            
def init_user_info_directory(df_users):
    for i in tqdm(range(len(df_users))):
        user_login = df_users.iloc[i]['login']
        user_directory_path = os.path.join('Database/UserData/', user_login)
        os.makedirs(user_directory_path, exist_ok=True)
        # saves the user's information
        user_info = df_users.iloc[i].to_dict()
        with open(os.path.join(user_directory_path, 'user_info.json'), 'w') as json_file:
            json.dump(user_info, json_file, indent=4)

In [21]:
def add_language_details(df_repos, github_request_machine):
    # Add language details
    df_repos['language_details'] = None
    df_repos.reset_index(drop=True, inplace=True)
    for i in range(len(df_repos)):
        repo_full_name = df_repos.iloc[i]['full_name']
        df_repos.at[i, 'language_details'] = github_request_machine.repository_service_endpoint(repo_full_name, 'languages', retry_after_rate_limit=True)
    return df_repos

def add_branch_details(df_repos, github_request_machine):
    # Add branches detail
    df_repos['num_branches'] = None
    df_repos.reset_index(drop=True, inplace=True)
    for i in range(len(df_repos)):
        repo_full_name = df_repos.iloc[i]['full_name']
        branches = github_request_machine.repository_service_endpoint(repo_full_name, 'branches', retry_after_rate_limit=True)
        df_repos.at[i, 'num_branches'] = len(branches)
    return df_repos

def add_contributor_details(df_repos, github_request_machine):
    # Add commit details
    df_repos['contributors'] = None
    df_repos.reset_index(drop=True, inplace=True)
    for i in range(len(df_repos)):
        repo_full_name = df_repos.iloc[i]['full_name']
        contributors = github_request_machine.repository_service_endpoint(repo_full_name, 'contributors', retry_after_rate_limit=True)
        df_repos.at[i, 'contributors'] = contributors
    return df_repos


def add_detail_to_single_user_python_repos(user_login, github_request_machine):
    user_directory_path = os.path.join('Database/UserData/', user_login)
    os.makedirs(user_directory_path, exist_ok=True)
    # construct the csv file path
    repo_csv_path = os.path.join(user_directory_path, 'repos.csv')
    if not os.path.exists(repo_csv_path):
        print(f"User {user_login} haven't run store_users_repos yet!")
        return
    # construct a new csv file for filtered python repos
    python_repo_csv_path = os.path.join(user_directory_path, 'python_repo_details.csv')
    # Cache the file
    if os.path.exists(python_repo_csv_path):
        return
    # find all the python "original" repos
    df_repos = pd.read_csv(repo_csv_path)
    df_repos = df_repos[(df_repos['fork'] == False) &                    # Non-forked
                            (df_repos['mirror_url'].isnull()) &          # Not a mirror
                            (df_repos['is_template'] == False) &         # Not a template
                            (df_repos['language'] == 'Python')]          # Language is Python
    
    # add details to them
    df_repos = add_language_details(df_repos, github_request_machine)
    df_repos = add_branch_details(df_repos, github_request_machine)
    df_repos = add_contributor_details(df_repos, github_request_machine)
    # save this to csv
    df_repos.to_csv(python_repo_csv_path, index=False)

def add_details_to_python_repos(df_users, github_request_machine):
    for i in tqdm(range(len(df_users))):
        user_login = df_users.iloc[i]['login']
        try:
            add_detail_to_single_user_python_repos(user_login, github_request_machine)
        except:
            print(f"Trouble: {i} {user_login}")
            continue

                 
        
        
        

In [25]:
def main(api_key, country, start_idx, end_idx):
    github_request_machine = GithubRequestMachine(api_key)

    # clip the df
    df_users = pd.read_csv("Database/TempData/GatherData/Filter_2_DecypherLocation/Results/users_filter_2.csv")
    df_users_country = df_users[df_users['country']== country]
    df_users_to_process = df_users_country[start_idx:end_idx]

    # process the clipped df
    init_user_info_directory(df_users_to_process)
    store_users_repos(df_users_to_process, github_request_machine)
    add_details_to_python_repos(df_users_to_process, github_request_machine)

from Utils.GithubAPI import GithubRequestMachine, RateLimitError
from Utils.Secrete import SecreteLoader

# Sam's Token: You can load other people
GITHUB_FINE_GRAINED_ACCESS_TOKEN = SecreteLoader().get_github_apikey('Sam')

main(GITHUB_FINE_GRAINED_ACCESS_TOKEN, 'United States', 0, 100)


100%|█████████████████████████████████████████████████████████████████████████████████| 100/100 [00:28<00:00,  3.49it/s]

<Response [451]>
Trouble: 71 philipperemy


In [ ]:
github_request_machine.check_rate_limit()